# 05k-b — Reassessment causale del fallimento autoregressivo

Il 05k ha fallito per tutti e tre i seed. Questo notebook non addestra e non seleziona modelli: applica cinque interventi congelati per distinguere drift del voltaggio, drift latente e ricorsione del decoder. Gli interventi teacher sono oracle diagnostici e non input di deployment.

## 1. Checkout riproducibile

In [ ]:
import importlib,json,os,shutil,subprocess,sys,zipfile,hashlib,time
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main')
ROOT=Path('/kaggle/working');WORKSPACE=ROOT/'hayflow_workspace';WORKSPACE.mkdir(parents=True,exist_ok=True);ELM_REPO=WORKSPACE/'elmneuron'
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'owned':str(ELM_REPO),'revision':REVISION})

## 2. Dipendenze e GPU

In [ ]:
run([sys.executable,'-m','pip','install','--quiet','numpy','pandas','h5py','pyarrow','pyyaml'])
import h5py,numpy as np,pandas as pd,pyarrow,torch,yaml
assert torch.cuda.is_available(),'Attiva una GPU Kaggle per il reassessment.'
print({'torch':torch.__version__,'cuda':torch.cuda.get_device_name(0)})

## 3. Input esatti e catena sperimentale

In [ ]:
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_size';stamp=str(source.stat().st_size)
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
def resolve_source(env,zip_name,marker):
 candidates=([Path(os.environ[env]).expanduser()] if os.environ.get(env) else [])+list(INPUT_ROOT.rglob(zip_name))+[p.parent for p in INPUT_ROOT.rglob(marker)]
 found=next((p.resolve() for p in candidates if p.exists()),None);assert found is not None,f'{zip_name} non trovato.';return found
def valid_base(path):return (Path(path)/'transition_dataset.h5').is_file() and (Path(path)/'targeted_pilot'/'candidate_trials.parquet').is_file()
base_candidates=([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else [])+[p.parent.parent for p in INPUT_ROOT.rglob('candidate_trials.parquet') if p.parent.name=='targeted_pilot'];BASE_SOURCE=next((p.resolve() for p in base_candidates if valid_base(p)),None);assert BASE_SOURCE is not None,'Dataset targeted v1.1 completo non trovato.'
TOPUP_SOURCE=resolve_source('HAYFLOW_TOPUP_V3','hayflow_bap_validation_support_topup_v3.zip','composite_dataset_manifest.json');TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow05kb_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE;manifests=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifests)==1,manifests;COMPOSITE_MANIFEST=manifests[0]
chain=[('CHECKPOINT_05B_SOURCE','HAYFLOW_05B_ARTIFACT','hayflow_hines_canary_v2.zip','canary_models.pt'),('ARTIFACT_05C_SOURCE','HAYFLOW_05C_ARTIFACT','hayflow_hines_causal_isolation.zip','checkpoint_forensics.json'),('ARTIFACT_05D_SOURCE','HAYFLOW_05D_ARTIFACT','hayflow_hines_residual_conditioning.zip','free_residual_report.json'),('ARTIFACT_05E_SOURCE','HAYFLOW_05E_ARTIFACT','hayflow_hines_segment_capacity.zip','capacity_probe_report.json'),('ARTIFACT_05F_SOURCE','HAYFLOW_05F_ARTIFACT','hayflow_hines_segment_micro_canary.zip','micro_canary_report.json'),('ARTIFACT_05G_SOURCE','HAYFLOW_05G_ARTIFACT','hayflow_hines_optimization_audit.zip','optimization_support.json'),('ARTIFACT_05H_SOURCE','HAYFLOW_05H_ARTIFACT','hayflow_hines_representation_forensics.zip','representation_forensics_config.json'),('ARTIFACT_05I_SOURCE','HAYFLOW_05I_ARTIFACT','hayflow_hines_state_normalization_repair.zip','state_normalization_repair_config.json'),('ARTIFACT_05IB_SOURCE','HAYFLOW_05IB_ARTIFACT','hayflow_hines_netcon_semantic_state_repair.zip','netcon_semantic_repair_config.json'),('ARTIFACT_05IC_SOURCE','HAYFLOW_05IC_ARTIFACT','hayflow_hines_synaptic_domain_repair.zip','synaptic_domain_repair_config.json'),('ARTIFACT_05J_SOURCE','HAYFLOW_05J_ARTIFACT','hayflow_hines_repaired_representation_recheck.zip','repaired_representation_recheck_config.json'),('ARTIFACT_05JB_SOURCE','HAYFLOW_05JB_ARTIFACT','hayflow_hines_repaired_representation_revision.zip','repaired_representation_revision_config.json'),('ARTIFACT_05JC_SOURCE','HAYFLOW_05JC_ARTIFACT','hayflow_hines_spatial_support_revision.zip','spatial_support_revision_config.json'),('ARTIFACT_05JD_SOURCE','HAYFLOW_05JD_ARTIFACT','hayflow_hines_trainable_topology_decoder_micro_canary.zip','trainable_topology_canary_config.json'),('ARTIFACT_05JE_SOURCE','HAYFLOW_05JE_ARTIFACT','hayflow_hines_architecture_reassessment.zip','architecture_reassessment_config.json'),('ARTIFACT_05JF_SOURCE','HAYFLOW_05JF_ARTIFACT','hayflow_hines_region_mechanism_expert_revision.zip','region_mechanism_expert_config.json'),('ARTIFACT_05JG_SOURCE','HAYFLOW_05JG_ARTIFACT','hayflow_hines_regenerative_state_decomposition.zip','state_target_decomposition_config.json'),('ARTIFACT_05JH_SOURCE','HAYFLOW_05JH_ARTIFACT','hayflow_hines_regenerative_support_expansion.zip','regenerative_support_expansion_config.json'),('ARTIFACT_05JI_SOURCE','HAYFLOW_05JI_ARTIFACT','hayflow_regenerative_confirmation_support.zip','confirmation_plan.json'),('ARTIFACT_05JJ_SOURCE','HAYFLOW_05JJ_ARTIFACT','hayflow_hines_regenerative_confirmation.zip','independent_confirmation_config.json'),('ARTIFACT_05JK_SOURCE','HAYFLOW_05JK_ARTIFACT','hayflow_hines_voltage_objective_reassessment.zip','voltage_objective_reassessment_config.json'),('ARTIFACT_05JL_SOURCE','HAYFLOW_05JL_ARTIFACT','hayflow_hines_residual_safety_gate.zip','residual_safety_gate_config.json'),('ARTIFACT_05JM_SOURCE','HAYFLOW_05JM_ARTIFACT','hayflow_regenerative_training_support.zip','acquisition_contract.json'),('ARTIFACT_05JN_SOURCE','HAYFLOW_05JN_ARTIFACT','hayflow_hines_regenerative_decoder_refit.zip','regenerative_decoder_refit_config.json'),('ARTIFACT_05JO_SOURCE','HAYFLOW_05JO_ARTIFACT','hayflow_hines_regenerative_fresh_test.zip','frozen_model_evaluation_config.json'),('ARTIFACT_05K_SOURCE','HAYFLOW_05K_ARTIFACT','hayflow_hines_frozen_candidate_micro_rollout.zip','frozen_candidate_micro_rollout_config.json')]
for variable,env,name,marker in chain:globals()[variable]=resolve_source(env,name,marker)
if CHECKPOINT_05B_SOURCE.name=='checkpoints':CHECKPOINT_05B_SOURCE=CHECKPOINT_05B_SOURCE.parent
def index_matches(path,expected):
 path=Path(path)
 try:
  if path.is_file():
   with zipfile.ZipFile(path) as archive:
    names=[name for name in archive.namelist() if name.replace('\\','/').endswith('artifact_index.json')];return len(names)==1 and hashlib.sha256(archive.read(names[0])).hexdigest()==expected
  return any(hashlib.sha256(candidate.read_bytes()).hexdigest()==expected for candidate in path.rglob('artifact_index.json'))
 except (OSError,zipfile.BadZipFile):return False
def exact_artifact(env,name,marker,expected):
 candidates=([Path(os.environ[env]).expanduser()] if os.environ.get(env) else [])+list(INPUT_ROOT.rglob(name))+[path.parent for path in INPUT_ROOT.rglob(marker)]
 valid=[path.resolve() for path in candidates if path.exists() and index_matches(path,expected)];assert valid,f'{name} non trovato oppure artifact_index non compatibile.';return valid[0]
from src.hayflow_model.hines_state_normalization_repair import EXPECTED_05H_INDEX_SHA256
from src.hayflow_model.hines_netcon_semantic_repair import EXPECTED_05I_INDEX_SHA256
from src.hayflow_model.hines_synaptic_domain_repair import EXPECTED_05IB_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_recheck import EXPECTED_05IC_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_revision import EXPECTED_05J_INDEX_SHA256
from src.hayflow_model.hines_spatial_support_revision import EXPECTED_05JB_INDEX_SHA256
from src.hayflow_model.hines_trainable_topology_canary import EXPECTED_05JC_INDEX_SHA256
from src.hayflow_model.hines_architecture_reassessment import EXPECTED_05JD_INDEX_SHA256
from src.hayflow_model.hines_region_mechanism_experts import EXPECTED_05JE_INDEX_SHA256
from src.hayflow_model.hines_regenerative_state_decomposition import EXPECTED_05JF_INDEX_SHA256
from src.hayflow_model.hines_regenerative_support_expansion import EXPECTED_05JG_INDEX_SHA256
from src.hayflow_model.hines_regenerative_confirmation import EXPECTED_05JH_INDEX_SHA256,EXPECTED_05JI_INDEX_SHA256
from src.hayflow_model.hines_voltage_objective_reassessment import EXPECTED_05JJ_INDEX_SHA256
from src.hayflow_model.hines_residual_safety_gate import EXPECTED_05JK_INDEX_SHA256
from src.hayflow_model.hines_regenerative_decoder_refit import EXPECTED_05JL_INDEX_SHA256,EXPECTED_05JM_INDEX_SHA256
from src.hayflow_model.hines_regenerative_fresh_test import EXPECTED_05JN_INDEX_SHA256
from src.hayflow_model.hines_frozen_candidate_micro_rollout import EXPECTED_05JO_INDEX_SHA256
from src.hayflow_model.hines_autoregressive_failure_reassessment import EXPECTED_05K_INDEX_SHA256
indexed=[('ARTIFACT_05H_SOURCE','HAYFLOW_05H_ARTIFACT','hayflow_hines_representation_forensics.zip','representation_forensics_config.json',EXPECTED_05H_INDEX_SHA256),('ARTIFACT_05I_SOURCE','HAYFLOW_05I_ARTIFACT','hayflow_hines_state_normalization_repair.zip','state_normalization_repair_config.json',EXPECTED_05I_INDEX_SHA256),('ARTIFACT_05IB_SOURCE','HAYFLOW_05IB_ARTIFACT','hayflow_hines_netcon_semantic_state_repair.zip','netcon_semantic_repair_config.json',EXPECTED_05IB_INDEX_SHA256),('ARTIFACT_05IC_SOURCE','HAYFLOW_05IC_ARTIFACT','hayflow_hines_synaptic_domain_repair.zip','synaptic_domain_repair_config.json',EXPECTED_05IC_INDEX_SHA256),('ARTIFACT_05J_SOURCE','HAYFLOW_05J_ARTIFACT','hayflow_hines_repaired_representation_recheck.zip','repaired_representation_recheck_config.json',EXPECTED_05J_INDEX_SHA256),('ARTIFACT_05JB_SOURCE','HAYFLOW_05JB_ARTIFACT','hayflow_hines_repaired_representation_revision.zip','repaired_representation_revision_config.json',EXPECTED_05JB_INDEX_SHA256),('ARTIFACT_05JC_SOURCE','HAYFLOW_05JC_ARTIFACT','hayflow_hines_spatial_support_revision.zip','spatial_support_revision_config.json',EXPECTED_05JC_INDEX_SHA256),('ARTIFACT_05JD_SOURCE','HAYFLOW_05JD_ARTIFACT','hayflow_hines_trainable_topology_decoder_micro_canary.zip','trainable_topology_canary_config.json',EXPECTED_05JD_INDEX_SHA256),('ARTIFACT_05JE_SOURCE','HAYFLOW_05JE_ARTIFACT','hayflow_hines_architecture_reassessment.zip','architecture_reassessment_config.json',EXPECTED_05JE_INDEX_SHA256),('ARTIFACT_05JF_SOURCE','HAYFLOW_05JF_ARTIFACT','hayflow_hines_region_mechanism_expert_revision.zip','region_mechanism_expert_config.json',EXPECTED_05JF_INDEX_SHA256),('ARTIFACT_05JG_SOURCE','HAYFLOW_05JG_ARTIFACT','hayflow_hines_regenerative_state_decomposition.zip','state_target_decomposition_config.json',EXPECTED_05JG_INDEX_SHA256),('ARTIFACT_05JH_SOURCE','HAYFLOW_05JH_ARTIFACT','hayflow_hines_regenerative_support_expansion.zip','regenerative_support_expansion_config.json',EXPECTED_05JH_INDEX_SHA256),('ARTIFACT_05JI_SOURCE','HAYFLOW_05JI_ARTIFACT','hayflow_regenerative_confirmation_support.zip','confirmation_plan.json',EXPECTED_05JI_INDEX_SHA256),('ARTIFACT_05JJ_SOURCE','HAYFLOW_05JJ_ARTIFACT','hayflow_hines_regenerative_confirmation.zip','independent_confirmation_config.json',EXPECTED_05JJ_INDEX_SHA256),('ARTIFACT_05JK_SOURCE','HAYFLOW_05JK_ARTIFACT','hayflow_hines_voltage_objective_reassessment.zip','voltage_objective_reassessment_config.json',EXPECTED_05JK_INDEX_SHA256),('ARTIFACT_05JL_SOURCE','HAYFLOW_05JL_ARTIFACT','hayflow_hines_residual_safety_gate.zip','residual_safety_gate_config.json',EXPECTED_05JL_INDEX_SHA256),('ARTIFACT_05JM_SOURCE','HAYFLOW_05JM_ARTIFACT','hayflow_regenerative_training_support.zip','acquisition_contract.json',EXPECTED_05JM_INDEX_SHA256),('ARTIFACT_05JN_SOURCE','HAYFLOW_05JN_ARTIFACT','hayflow_hines_regenerative_decoder_refit.zip','regenerative_decoder_refit_config.json',EXPECTED_05JN_INDEX_SHA256),('ARTIFACT_05JO_SOURCE','HAYFLOW_05JO_ARTIFACT','hayflow_hines_regenerative_fresh_test.zip','frozen_model_evaluation_config.json',EXPECTED_05JO_INDEX_SHA256),('ARTIFACT_05K_SOURCE','HAYFLOW_05K_ARTIFACT','hayflow_hines_frozen_candidate_micro_rollout.zip','frozen_candidate_micro_rollout_config.json',EXPECTED_05K_INDEX_SHA256)]
for variable,env,name,marker,expected in indexed:globals()[variable]=exact_artifact(env,name,marker,expected)
print({'base':str(BASE_SOURCE),'05j-o':str(ARTIFACT_05JO_SOURCE),'05k':str(ARTIFACT_05K_SOURCE)})

## 4. Bundle e sessione diagnostica congelata

In [ ]:
from IPython.display import display
from src.hayflow_data import prepare_composite_flowmap_bundle
started,last={},{}
def hash_progress(name,done,total):
 now=time.monotonic();started.setdefault(name,now);pct=int(100*done/total)
 if pct>=last.get(name,-5)+5 or done==total:
  eta=(total-done)/max(done/max(now-started[name],1e-9),1e-9);print(f'[HayFlow 05k-b][SHA-256 {name}] {pct}% ETA {eta/60:.1f} min',flush=True);last[name]=pct
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880
from src.hayflow_model import *
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_hines_autoregressive_failure_reassessment');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
FRESH_ROOT,fresh_result,fresh_contract=verified_fresh_test_artifact_root(ARTIFACT_05JO_SOURCE,'/kaggle/working/.05kb_input_cache/05jo')
verified_05k_root,verified_05k_report,verified_05k_contract=verified_micro_rollout_artifact_root(ARTIFACT_05K_SOURCE,'/kaggle/working/.05kb_input_cache/05k')
def config(name):return yaml.safe_load((ELM_REPO/'configs/hayflow'/name).read_text())
b=config('hayflow_hines_optimization_audit.yml');f=config('hayflow_hines_representation_forensics.yml');r=config('hayflow_hines_state_normalization_repair.yml');n=config('hayflow_hines_netcon_semantic_repair.yml');d=config('hayflow_hines_synaptic_domain_repair.yml');rc=config('hayflow_hines_repaired_representation_recheck.yml');rv=config('hayflow_hines_repaired_representation_revision.yml');sp=config('hayflow_hines_spatial_support_revision.yml');tp=config('hayflow_hines_trainable_topology_canary.yml');ra=config('hayflow_hines_architecture_reassessment.yml');ex=config('hayflow_hines_region_mechanism_experts.yml');dc=config('hayflow_hines_regenerative_state_decomposition.yml');se=config('hayflow_hines_regenerative_support_expansion.yml');cf=config('hayflow_hines_regenerative_confirmation.yml');oa=config('hayflow_hines_voltage_objective_reassessment.yml');sg=config('hayflow_hines_residual_safety_gate.yml');rf=config('hayflow_hines_regenerative_decoder_refit.yml');fr=config('hayflow_regenerative_fresh_test.yml');mr=config('hayflow_hines_frozen_candidate_micro_rollout.yml');ar=config('hayflow_hines_autoregressive_failure_reassessment.yml')
model_config=HinesPrototypeExperimentConfig.from_mapping(b['model_experiment']);isolation_config=HinesIsolationConfig.from_mapping(b['isolation']);conditioning_config=HinesConditioningConfig.from_mapping(b['conditioning']);capacity_config=HinesCapacityConfig.from_mapping(b['capacity']);canary_config=HinesSegmentCanaryConfig.from_mapping(b['micro_canary']);audit_config=HinesOptimizationAuditConfig.from_mapping(b['optimization_audit']);representation_config=HinesRepresentationForensicsConfig.from_mapping(f['representation_forensics']);repair_config=HinesStateNormalizationRepairConfig.from_mapping(r['state_normalization_repair']);netcon_config=HinesNetConSemanticRepairConfig.from_mapping(n['netcon_semantic_repair']);domain_config=HinesSynapticDomainRepairConfig.from_mapping(d['synaptic_domain_repair']);recheck_config=HinesRepairedRepresentationRecheckConfig.from_mapping(rc['repaired_representation_recheck']);revision_config=HinesRepairedRepresentationRevisionConfig.from_mapping(rv['repaired_representation_revision']);spatial_config=HinesSpatialSupportRevisionConfig.from_mapping(sp['spatial_support_revision']);topology_config=HinesTrainableTopologyCanaryConfig.from_mapping(tp['trainable_topology_canary']);reassessment_config=HinesArchitectureReassessmentConfig.from_mapping(ra['architecture_reassessment']);expert_config=HinesRegionMechanismExpertConfig.from_mapping(ex['region_mechanism_experts']);decomposition_config=HinesRegenerativeStateDecompositionConfig.from_mapping(dc['regenerative_state_decomposition']);support_expansion_config=HinesRegenerativeSupportExpansionConfig.from_mapping(se['regenerative_support_expansion']);confirmation_config=HinesRegenerativeConfirmationConfig.from_mapping(cf['independent_confirmation']);objective_config=HinesVoltageObjectiveReassessmentConfig.from_mapping(oa['voltage_objective_reassessment']);safety_config=HinesResidualSafetyGateConfig.from_mapping(sg['residual_safety_gate']);refit_config=HinesRegenerativeDecoderRefitConfig.from_mapping(rf['regenerative_decoder_refit']);fresh_config=HinesRegenerativeFreshTestConfig.from_mapping(fr['frozen_model_gate']);micro_config=HinesFrozenCandidateMicroRolloutConfig.from_mapping(mr['frozen_candidate_micro_rollout']);autoregressive_config=HinesAutoregressiveFailureReassessmentConfig.from_mapping(ar['autoregressive_failure_reassessment'])
session=HinesAutoregressiveFailureReassessment(bundle,OUTPUT_DIR,model_config,isolation_config,conditioning_config,capacity_config,canary_config,audit_config,representation_config,CHECKPOINT_05B_SOURCE,ARTIFACT_05C_SOURCE,ARTIFACT_05D_SOURCE,ARTIFACT_05E_SOURCE,ARTIFACT_05F_SOURCE,ARTIFACT_05G_SOURCE,repair_config=repair_config,artifact_05h_source=ARTIFACT_05H_SOURCE,netcon_config=netcon_config,artifact_05i_source=ARTIFACT_05I_SOURCE,domain_config=domain_config,artifact_05ib_source=ARTIFACT_05IB_SOURCE,recheck_config=recheck_config,artifact_05ic_source=ARTIFACT_05IC_SOURCE,revision_config=revision_config,artifact_05j_source=ARTIFACT_05J_SOURCE,spatial_config=spatial_config,artifact_05jb_source=ARTIFACT_05JB_SOURCE,topology_config=topology_config,artifact_05jc_source=ARTIFACT_05JC_SOURCE,reassessment_config=reassessment_config,artifact_05jd_source=ARTIFACT_05JD_SOURCE,expert_config=expert_config,artifact_05je_source=ARTIFACT_05JE_SOURCE,decomposition_config=decomposition_config,artifact_05jf_source=ARTIFACT_05JF_SOURCE,support_expansion_config=support_expansion_config,artifact_05jg_source=ARTIFACT_05JG_SOURCE,confirmation_config=confirmation_config,artifact_05jh_source=ARTIFACT_05JH_SOURCE,artifact_05ji_source=ARTIFACT_05JI_SOURCE,objective_config=objective_config,artifact_05jj_source=ARTIFACT_05JJ_SOURCE,safety_config=safety_config,artifact_05jk_source=ARTIFACT_05JK_SOURCE,refit_config=refit_config,artifact_05jl_source=ARTIFACT_05JL_SOURCE,artifact_05jm_source=ARTIFACT_05JM_SOURCE,fresh_test_config=fresh_config,artifact_05jn_source=ARTIFACT_05JN_SOURCE,fresh_dataset_root=FRESH_ROOT,micro_rollout_config=micro_config,artifact_05jo_source=ARTIFACT_05JO_SOURCE,failure_reassessment_config=autoregressive_config,artifact_05k_source=ARTIFACT_05K_SOURCE,code_revision=REVISION)
prepare=session.prepare_autoregressive_failure_reassessment();display({'revision':REVISION,'05k_diagnosis':verified_05k_report['diagnosis'],'fresh_store_valid':prepare['fresh_store']['valid'],'interventions':prepare['reassessment']['intervention_modes'],'retraining':prepare['retraining_performed']});assert prepare['valid'] and prepare['fresh_store']['valid'] and not prepare['retraining_performed']

## 5. Ricostruzione bit-compatible dei checkpoint congelati

In [ ]:
session.apply_verified_synaptic_domain_normalizer();session.build_expanded_train_support();session.prepare_expanded_spatial_features();design=session.prepare_topology_canary_designs();session.fit_fixed_tree_ridge_baseline();reconstruction=session.reconstruct_frozen_checkpoints(metric_atol=expert_config.checkpoint_reconstruction_metric_atol);session.build_regenerative_support();session.prepare_expanded_regenerative_roles();expanded=session.reconstruct_expanded_direct_tree_ensemble();external=session.prepare_external_confirmation_roles();roles=session.prepare_refit_roles()
display({'design_valid':design['valid'],'checkpoint_reconstruction_valid':reconstruction['valid'],'expanded_valid':expanded['valid'],'development_valid':external['valid'],'refit_roles_valid':roles['valid']});assert design['valid'] and reconstruction['valid'] and expanded['valid'] and external['valid'] and roles['valid']

## 6. Matrice di interventi causali e diagnosi

In [ ]:
report=session.evaluate_failure_interventions();rows=[]
for seed,by_mode in report['metrics'].items():
 for mode,by_horizon in by_mode.items():
  metrics=by_horizon['8'];rows.append({'seed':int(seed),'mode':mode,'rmse_8ms_mv':metrics['endpoint_voltage_rmse_mv'],'drift_8ms_mv':metrics['endpoint_mean_drift_mv'],'reduction_vs_closed':metrics['error_reduction_vs_closed_loop_fraction'],'retention':metrics['median_branching_retention'],'physical_violations':metrics['physical_voltage_violation_count']})
display(pd.DataFrame(rows).sort_values(['mode','seed']));display({'valid':report['valid'],'target_reproduction_error_mv':report['target_reproduction_error_mv'],'attribution':report['attribution'],'retraining':report['retraining_performed'],'model_or_training_authorized':report['model_or_training_authorized']});assert report['valid'] and not report['retraining_performed'] and not report['model_or_training_authorized']
final_report=session.finalize_failure_reassessment(report);display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'candidate_reinstated':final_report['candidate_reinstated'],'training_authorized':final_report['training_authorized'],'new_sealed_test_required':final_report['future_candidate_requires_new_sealed_test'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['candidate_reinstated'] and not final_report['training_authorized']

## 7. Crea e scarica lo ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_hines_autoregressive_failure_reassessment','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})